# Repeat Purchase Prediction — Phase 1: Dataset Inspection & Quality Check

**Goal:** Load the Online Retail II dataset, inspect its structure, and document 
data quality issues before deciding on a cleaning plan.

**Dataset:** Online Retail II (UCI/Kaggle) — UK-based online gift retailer, 
Dec 2009–Dec 2011, ~1.07M transaction rows across two sheets.

In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

# Load both sheets and combine
df_1 = pd.read_excel('../data/Raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2 = pd.read_excel('../data/Raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')

df = pd.concat([df_1, df_2], ignore_index=True)

print("Combined shape:", df.shape)
df.head()

Combined shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
df.to_pickle('../data/Processed/raw_combined.pkl')

In [6]:
import pandas as pd
df = pd.read_pickle('../data/Processed/raw_combined.pkl')
print(df.shape)

(1067371, 8)


## Initial Structure Check

In [7]:
print("Columns:", list(df.columns))
print()
print("Data types:")
print(df.dtypes)
print()
print("Date range:", df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())
print()
print("Unique customers:", df['Customer ID'].nunique())
print("Unique countries:", df['Country'].nunique())

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00

Unique customers: 5942
Unique countries: 43


## Data Quality Check

Checking for the known issues in this dataset before deciding how to handle each:
- Missing Customer IDs
- Missing product descriptions
- Cancelled orders (Invoice starting with "C")
- Negative quantities (returns/adjustments)
- Negative or zero prices (data errors)
- Duplicate rows

In [8]:
print("Missing values per column:")
print(df.isnull().sum())
print()

missing_id_pct = df['Customer ID'].isnull().mean() * 100
print(f"Missing Customer ID: {missing_id_pct:.2f}%")
print()

cancelled = df['Invoice'].astype(str).str.startswith('C')
print(f"Cancelled invoices: {cancelled.sum()} ({cancelled.mean()*100:.2f}%)")
print()

print("Negative Quantity rows:", (df['Quantity'] < 0).sum())
print("Negative/zero Price rows:", (df['Price'] <= 0).sum())
print()

print("Duplicate rows:", df.duplicated().sum())

Missing values per column:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Missing Customer ID: 22.77%

Cancelled invoices: 19494 (1.83%)

Negative Quantity rows: 22950
Negative/zero Price rows: 6207

Duplicate rows: 34335


## Documented Cleaning Decisions

Based on the checks above:

1. **Missing Customer ID (~23% of rows):** These are likely guest checkouts with 
   no account. Since our analysis is customer-level, these rows cannot be used 
   and will be dropped. This is a known, unavoidable limitation — documented in 
   the README, not silently fixed.

2. **Cancelled orders (Invoice starting with "C"):** Dropped entirely from the 
   dataset. These represent order cancellations, not completed purchases, and 
   are not relevant to predicting repeat purchase behavior in this scope. 
   (Considered keeping as a "customer ever cancelled" feature, but excluded to 
   keep the project scope focused.)

3. **Negative quantities:** Represent returns/adjustments, not real purchases — 
   will be excluded from purchase-based features.

4. **Negative/zero prices:** Likely data entry errors — will be excluded.

5. **Duplicate rows:** Will be dropped as exact duplicates carry no extra signal.

6. **Missing Descriptions:** Minor (~0.4% of rows) — only relevant if using 
   product-category features; won't block core analysis.

## Applying the Cleaning Steps

Applying each decision documented above, in order, and checking the row count 
after each step so we can see the impact of each individual decision.

In [9]:
print("Before:", df.shape)

df_clean = df.dropna(subset=['Customer ID']).copy()

print("After dropping missing Customer ID:", df_clean.shape)

Before: (1067371, 8)
After dropping missing Customer ID: (824364, 8)


In [10]:
before = df_clean.shape[0]

df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]

print(f"Dropped {before - df_clean.shape[0]} cancelled order rows")
print("Shape now:", df_clean.shape)

Dropped 18744 cancelled order rows
Shape now: (805620, 8)


In [11]:
before = df_clean.shape[0]

df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

print(f"Dropped {before - df_clean.shape[0]} rows with non-positive quantity/price")
print("Shape now:", df_clean.shape)

Dropped 71 rows with non-positive quantity/price
Shape now: (805549, 8)


In [12]:
before = df_clean.shape[0]

df_clean = df_clean.drop_duplicates()

print(f"Dropped {before - df_clean.shape[0]} duplicate rows")
print("Shape now:", df_clean.shape)

Dropped 26124 duplicate rows
Shape now: (779425, 8)


In [13]:
print("Final cleaned shape:", df_clean.shape)
print()
print("Remaining missing values:")
print(df_clean.isnull().sum())
print()
print("Unique customers remaining:", df_clean['Customer ID'].nunique())
print("Date range:", df_clean['InvoiceDate'].min(), "to", df_clean['InvoiceDate'].max())

Final cleaned shape: (779425, 8)

Remaining missing values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

Unique customers remaining: 5878
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [14]:
df_clean.to_pickle('../data/Processed/cleaned_transactions.pkl')
print("Saved cleaned dataset.")

Saved cleaned dataset.


# Phase 2: Defining the Repeat Purchase Label

**Goal:** For each customer, determine their first order date, then check 
whether they placed another order within 90 days of that first order.

**Important bias check:** customers whose first order falls too close to the 
end of the dataset (Dec 2011) won't have had a full 90-day window to show 
repeat behavior yet. Including them would unfairly label them "0" (no repeat) 
just because we ran out of time to observe them — this would bias the model. 
These customers must be excluded from the labeled dataset.

In [15]:
# Each row is a product line item — get unique order-level records first
# (one row per Invoice, using the invoice date and customer)
orders = df_clean.groupby(['Customer ID', 'Invoice']).agg(
    order_date=('InvoiceDate', 'min')
).reset_index()

print("Total distinct orders:", orders.shape[0])
orders.head()

Total distinct orders: 36969


,Customer ID,Invoice,order_date
0,12346.0,491725,2009-12-14 08:34:00
1,12346.0,491742,2009-12-14 11:00:00
2,12346.0,491744,2009-12-14 11:02:00
3,12346.0,492718,2009-12-18 10:47:00
4,12346.0,492722,2009-12-18 10:55:00


In [16]:
first_order = orders.groupby('Customer ID')['order_date'].min().reset_index()
first_order.columns = ['Customer ID', 'first_order_date']

print("Customers with a first order date:", first_order.shape[0])
first_order.head()

Customers with a first order date: 5878


,Customer ID,first_order_date
0,12346.0,2009-12-14 08:34:00
1,12347.0,2010-10-31 14:20:00
2,12348.0,2010-09-27 14:59:00
3,12349.0,2010-04-29 13:20:00
4,12350.0,2011-02-02 16:01:00


In [17]:
# Merge first order date back into the full orders table
orders_merged = orders.merge(first_order, on='Customer ID')

# Flag orders that happened AFTER the first order (not the first order itself)
orders_merged['days_since_first'] = (
    orders_merged['order_date'] - orders_merged['first_order_date']
).dt.days

# A repeat purchase = any order with days_since_first between 1 and 90
repeat_flag = orders_merged[
    (orders_merged['days_since_first'] > 0) & 
    (orders_merged['days_since_first'] <= 90)
].groupby('Customer ID').size().reset_index(name='repeat_orders_count')

first_order = first_order.merge(repeat_flag, on='Customer ID', how='left')
first_order['repeat_orders_count'] = first_order['repeat_orders_count'].fillna(0)
first_order['repeat_purchase'] = (first_order['repeat_orders_count'] > 0).astype(int)

print(first_order['repeat_purchase'].value_counts())
first_order.head()

repeat_purchase
0    3300
1    2578
Name: count, dtype: int64


,Customer ID,first_order_date,repeat_orders_count,repeat_purchase
0,12346.0,2009-12-14 08:34:00,7.0,1
1,12347.0,2010-10-31 14:20:00,2.0,1
2,12348.0,2010-09-27 14:59:00,1.0,1
3,12349.0,2010-04-29 13:20:00,1.0,1
4,12350.0,2011-02-02 16:01:00,0.0,0


## Excluding Customers With an Incomplete Observation Window

Customers whose first order was within the last 90 days of the dataset 
(i.e., after Sep 10, 2011, since the dataset ends Dec 9, 2011) haven't had 
a fair chance to show repeat behavior. These must be excluded to avoid bias.

In [18]:
max_date = df_clean['InvoiceDate'].max()
cutoff_date = max_date - pd.Timedelta(days=90)

print("Dataset max date:", max_date)
print("Cutoff date for valid first orders:", cutoff_date)

before = first_order.shape[0]
first_order = first_order[first_order['first_order_date'] <= cutoff_date]

print(f"Excluded {before - first_order.shape[0]} customers with incomplete observation window")
print("Remaining customers for modeling:", first_order.shape[0])
print()
print("Final repeat purchase distribution:")
print(first_order['repeat_purchase'].value_counts(normalize=True))

Dataset max date: 2011-12-09 12:50:00
Cutoff date for valid first orders: 2011-09-10 12:50:00
Excluded 597 customers with incomplete observation window
Remaining customers for modeling: 5281

Final repeat purchase distribution:
repeat_purchase
0    0.553304
1    0.446696
Name: proportion, dtype: float64


# Phase 3: Feature Engineering

**Goal:** Build features using ONLY information available at the time of the 
customer's first order — this matches a real business scenario where you'd 
need to predict repeat purchase behavior right after someone's first order, 
before knowing anything else about them.

Features to build:
- Total spend on first order
- Total quantity of items purchased
- Number of distinct products (variety)
- Average price per item
- Day of week / month of first order (timing/seasonality)
- Country

In [19]:
# Add a line-level total (Quantity * Price) to df_clean
df_clean['line_total'] = df_clean['Quantity'] * df_clean['Price']

# Merge first_order_date into the cleaned transaction data
df_features = df_clean.merge(
    first_order[['Customer ID', 'first_order_date']], 
    on='Customer ID'
)

# Keep only the line items that belong to the customer's FIRST order 
# (same invoice date/time as their first order)
first_order_items = df_features[
    df_features['InvoiceDate'] == df_features['first_order_date']
]

print("Line items belonging to first orders:", first_order_items.shape[0])
first_order_items.head()

Line items belonging to first orders: 126837


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,line_total,first_order_date
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009-12-01 07:45:00
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12-01 07:45:00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12-01 07:45:00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009-12-01 07:45:00
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009-12-01 07:45:00


In [20]:
features = first_order_items.groupby('Customer ID').agg(
    total_spend=('line_total', 'sum'),
    total_quantity=('Quantity', 'sum'),
    num_distinct_products=('StockCode', 'nunique'),
    avg_price_per_item=('Price', 'mean'),
    first_order_date=('first_order_date', 'first'),
    country=('Country', 'first')
).reset_index()

print("Feature table shape:", features.shape)
features.head()

Feature table shape: (5281, 7)


,Customer ID,total_spend,total_quantity,num_distinct_products,avg_price_per_item,first_order_date,country
0,12346.0,45.00,10,1,4.500000,2009-12-14 08:34:00,United Kingdom
1,12347.0,611.53,509,40,1.834000,2010-10-31 14:20:00,Iceland
2,12348.0,222.16,373,20,0.719500,2010-09-27 14:59:00,Finland
3,12349.0,1068.52,473,46,4.230000,2010-04-29 13:20:00,Italy
4,12350.0,334.40,197,17,3.841176,2011-02-02 16:01:00,Norway


In [21]:
features['order_month'] = features['first_order_date'].dt.month
features['order_dayofweek'] = features['first_order_date'].dt.dayofweek  # 0=Monday

features.head()

,Customer ID,total_spend,total_quantity,num_distinct_products,avg_price_per_item,first_order_date,country,order_month,order_dayofweek
0,12346.0,45.00,10,1,4.500000,2009-12-14 08:34:00,United Kingdom,12,0
1,12347.0,611.53,509,40,1.834000,2010-10-31 14:20:00,Iceland,10,6
2,12348.0,222.16,373,20,0.719500,2010-09-27 14:59:00,Finland,9,0
3,12349.0,1068.52,473,46,4.230000,2010-04-29 13:20:00,Italy,4,3
4,12350.0,334.40,197,17,3.841176,2011-02-02 16:01:00,Norway,2,2


In [22]:
final_dataset = features.merge(
    first_order[['Customer ID', 'repeat_purchase']], 
    on='Customer ID'
)

print("Final modeling dataset shape:", final_dataset.shape)
final_dataset.head()

Final modeling dataset shape: (5281, 10)


,Customer ID,total_spend,total_quantity,num_distinct_products,avg_price_per_item,first_order_date,country,order_month,order_dayofweek,repeat_purchase
0,12346.0,45.00,10,1,4.500000,2009-12-14 08:34:00,United Kingdom,12,0,1
1,12347.0,611.53,509,40,1.834000,2010-10-31 14:20:00,Iceland,10,6,1
2,12348.0,222.16,373,20,0.719500,2010-09-27 14:59:00,Finland,9,0,1
3,12349.0,1068.52,473,46,4.230000,2010-04-29 13:20:00,Italy,4,3,1
4,12350.0,334.40,197,17,3.841176,2011-02-02 16:01:00,Norway,2,2,0


In [23]:
print(final_dataset.describe())
print()
print("Missing values:")
print(final_dataset.isnull().sum())
print()
print("Top countries:")
print(final_dataset['country'].value_counts().head(10))

        Customer ID   total_spend  total_quantity  num_distinct_products  \
count   5281.000000   5281.000000     5281.000000            5281.000000   
mean   15324.100170    411.391533      258.953986              23.662185   
min    12346.000000      1.300000        1.000000               1.000000   
25%    13850.000000    165.440000       79.000000               9.000000   
50%    15313.000000    293.150000      150.000000              18.000000   
75%    16807.000000    446.490000      277.000000              30.000000   
max    18287.000000  33167.800000    87167.000000             250.000000   
std     1712.323865    756.985644     1294.580308              22.451916   

       avg_price_per_item               first_order_date  order_month  \
count         5281.000000                           5281  5281.000000   
mean             8.939948  2010-07-05 05:33:59.318310912     6.695891   
min              0.151333            2009-12-01 07:45:00     1.000000   
25%              2.1378

In [24]:
# Looking at the customer with the highest avg_price_per_item
print("Highest avg_price_per_item customer:")
print(final_dataset.sort_values('avg_price_per_item', ascending=False).head(3))
print()

# Looking at the customer with the highest total_quantity
print("Highest total_quantity customer:")
print(final_dataset.sort_values('total_quantity', ascending=False).head(3))
print()

# Looking at the raw transaction rows for the highest avg_price_per_item customer
outlier_id = final_dataset.sort_values('avg_price_per_item', ascending=False).iloc[0]['Customer ID']
print(f"Raw transactions for Customer ID {outlier_id}:")
print(df_clean[df_clean['Customer ID'] == outlier_id][['StockCode', 'Description', 'Quantity', 'Price', 'InvoiceDate']])

Highest avg_price_per_item customer:
      Customer ID  total_spend  total_quantity  num_distinct_products  \
499       12918.0     10953.50               1                      1   
3027      15760.0      6958.17               1                      1   
2178      14802.0      1343.44               1                      1   

      avg_price_per_item    first_order_date         country  order_month  \
499             10953.50 2010-03-23 15:22:00  United Kingdom            3   
3027             6958.17 2010-03-19 11:35:00          Norway            3   
2178             1343.44 2010-09-27 16:32:00  United Kingdom            9   

      order_dayofweek  repeat_purchase  
499                 1                0  
3027                4                0  
2178                0                0  

Highest total_quantity customer:
      Customer ID  total_spend  total_quantity  num_distinct_products  \
1174      13687.0     11880.84           87167                     45   
5067      18052.0

In [25]:
# Common known special/non-product codes in this dataset
special_codes = df_clean[df_clean['StockCode'].astype(str).str.upper().isin(
    ['M', 'MANUAL', 'POST', 'D', 'DOT', 'BANK CHARGES', 'C2', 'CRUK', 'PADS', 'AMAZONFEE']
)]
print("Rows with special/non-product codes:", special_codes.shape[0])
print(special_codes['StockCode'].value_counts())

Rows with special/non-product codes: 2801
StockCode
POST            1803
M                681
C2               248
BANK CHARGES      31
PADS              17
DOT               16
D                  5
Name: count, dtype: int64


**Update to Cleaning Decisions:** During feature engineering, discovered that 
StockCode contains non-product administrative codes (e.g., 'M' for Manual 
adjustments, 'POST' for postage, 'D' for Discount, 'BANK CHARGES', 'DOT', 
'C2', 'PADS') — these represent adjustments or fees, not real product 
purchases, and were skewing spend-based features. These 2,801 rows are 
excluded from the cleaned dataset.

In [26]:
special_codes_list = ['M', 'MANUAL', 'POST', 'D', 'DOT', 'BANK CHARGES', 'C2', 'CRUK', 'PADS', 'AMAZONFEE']

before = df_clean.shape[0]

df_clean = df_clean[~df_clean['StockCode'].astype(str).str.upper().isin(special_codes_list)]

print(f"Dropped {before - df_clean.shape[0]} rows with non-product codes")
print("Shape now:", df_clean.shape)

# Re-save the cleaned dataset with this fix included
df_clean.to_pickle('../data/Processed/cleaned_transactions.pkl')
print("Re-saved cleaned dataset.")

Dropped 2801 rows with non-product codes
Shape now: (776624, 9)
Re-saved cleaned dataset.


In [27]:
print("Shape after removing special codes:", df_clean.shape)

Shape after removing special codes: (776624, 9)


In [28]:
print(df_clean['Country'].value_counts().head(10))
print()
print("UK % of transactions:", round((df_clean['Country'] == 'United Kingdom').mean() * 100, 2))

Country
United Kingdom    699633
Germany            15790
EIRE               15360
France             13033
Netherlands         4981
Spain               3563
Switzerland         2951
Belgium             2911
Portugal            2296
Australia           1783
Name: count, dtype: int64

UK % of transactions: 90.09


## Country Feature — Scoping Decision

The dataset is heavily skewed toward UK customers (~90%). Using all 43 raw 
country values as a categorical feature would create many near-empty 
categories, adding noise rather than useful signal, and risking overfitting 
to rare countries with very few customers.

**Decision:** Simplify country into a binary feature — `is_uk` (1 if UK, 
0 otherwise) — capturing the one geographic distinction that has enough 
sample size to be statistically meaningful in this dataset.

In [29]:
df_clean['line_total'] = df_clean['Quantity'] * df_clean['Price']

df_features = df_clean.merge(
    first_order[['Customer ID', 'first_order_date']], 
    on='Customer ID'
)

first_order_items = df_features[
    df_features['InvoiceDate'] == df_features['first_order_date']
]

print("Line items belonging to first orders:", first_order_items.shape[0])

Line items belonging to first orders: 126400


In [30]:
features = first_order_items.groupby('Customer ID').agg(
    total_spend=('line_total', 'sum'),
    total_quantity=('Quantity', 'sum'),
    num_distinct_products=('StockCode', 'nunique'),
    avg_price_per_item=('Price', 'mean'),
    first_order_date=('first_order_date', 'first'),
    country=('Country', 'first')
).reset_index()

features['is_uk'] = (features['country'] == 'United Kingdom').astype(int)

print("Feature table shape:", features.shape)
features.head()

Feature table shape: (5252, 8)


,Customer ID,total_spend,total_quantity,num_distinct_products,avg_price_per_item,first_order_date,country,is_uk
0,12346.0,45.00,10,1,4.500000,2009-12-14 08:34:00,United Kingdom,1
1,12347.0,611.53,509,40,1.834000,2010-10-31 14:20:00,Iceland,0
2,12348.0,221.16,372,19,0.704737,2010-09-27 14:59:00,Finland,0
3,12349.0,1068.52,473,46,4.230000,2010-04-29 13:20:00,Italy,0
4,12350.0,294.40,196,16,1.581250,2011-02-02 16:01:00,Norway,0


In [31]:
features['order_month'] = features['first_order_date'].dt.month
features['order_dayofweek'] = features['first_order_date'].dt.dayofweek

features.head()

,Customer ID,total_spend,total_quantity,num_distinct_products,avg_price_per_item,first_order_date,country,is_uk,order_month,order_dayofweek
0,12346.0,45.00,10,1,4.500000,2009-12-14 08:34:00,United Kingdom,1,12,0
1,12347.0,611.53,509,40,1.834000,2010-10-31 14:20:00,Iceland,0,10,6
2,12348.0,221.16,372,19,0.704737,2010-09-27 14:59:00,Finland,0,9,0
3,12349.0,1068.52,473,46,4.230000,2010-04-29 13:20:00,Italy,0,4,3
4,12350.0,294.40,196,16,1.581250,2011-02-02 16:01:00,Norway,0,2,2


In [32]:
final_dataset = features.merge(
    first_order[['Customer ID', 'repeat_purchase']], 
    on='Customer ID'
)

print("Final modeling dataset shape:", final_dataset.shape)
final_dataset.head()

Final modeling dataset shape: (5252, 11)


,Customer ID,total_spend,total_quantity,num_distinct_products,avg_price_per_item,first_order_date,country,is_uk,order_month,order_dayofweek,repeat_purchase
0,12346.0,45.00,10,1,4.500000,2009-12-14 08:34:00,United Kingdom,1,12,0,1
1,12347.0,611.53,509,40,1.834000,2010-10-31 14:20:00,Iceland,0,10,6,1
2,12348.0,221.16,372,19,0.704737,2010-09-27 14:59:00,Finland,0,9,0,1
3,12349.0,1068.52,473,46,4.230000,2010-04-29 13:20:00,Italy,0,4,3,1
4,12350.0,294.40,196,16,1.581250,2011-02-02 16:01:00,Norway,0,2,2,0


In [34]:
print(final_dataset.describe())
print()
print("Missing values:")
print(final_dataset.isnull().sum())
print()
print("is_uk distribution:")
print(final_dataset['is_uk'].value_counts())

        Customer ID   total_spend  total_quantity  num_distinct_products  \
count   5252.000000   5252.000000     5252.000000            5252.000000   
mean   15327.373001    404.671013      260.077875              23.713062   
min    12346.000000      1.300000        1.000000               1.000000   
25%    13850.750000    164.962500       80.000000               9.000000   
50%    15318.000000    289.605000      151.000000              18.000000   
75%    16812.250000    440.090000      277.000000              30.250000   
max    18287.000000  33167.800000    87167.000000             250.000000   
std     1713.038770    737.403046     1297.980804              22.430657   

       avg_price_per_item               first_order_date        is_uk  \
count         5252.000000                           5252  5252.000000   
mean             3.751612  2010-07-05 07:15:57.475247616     0.911843   
min              0.151333            2009-12-01 07:45:00     0.000000   
25%              2.0945

## Handling Remaining Skew — Log Transformation

`total_spend` and `total_quantity` remain right-skewed even after removing 
data errors (e.g., one legitimate wholesale buyer with total_quantity of 
87,167). Rather than removing this real customer, we apply a log 
transformation to reduce the influence of extreme values — standard practice 
for skewed monetary/count data, especially important for scale-sensitive 
models like Logistic Regression. Both original and log-transformed versions 
are kept, so tree-based models (Random Forest, XGBoost) can use either.

In [35]:
import numpy as np

final_dataset['log_total_spend'] = np.log1p(final_dataset['total_spend'])
final_dataset['log_total_quantity'] = np.log1p(final_dataset['total_quantity'])

final_dataset[['total_spend', 'log_total_spend', 'total_quantity', 'log_total_quantity']].describe()

,total_spend,log_total_spend,total_quantity,log_total_quantity
count,5252.000000,5252.000000,5252.000000,5252.000000
mean,404.671013,5.613202,260.077875,4.961943
std,737.403046,0.856765,1297.980804,1.069673
min,1.300000,0.832909,1.000000,0.693147
25%,164.962500,5.111762,80.000000,4.394449
50%,289.605000,5.671965,151.000000,5.023881
75%,440.090000,6.089249,277.000000,5.627621
max,33167.800000,10.409365,87167.000000,11.375593


In [36]:
final_dataset.to_pickle('../data/Processed/modeling_dataset.pkl')
print("Saved final modeling dataset.")
print("Final shape:", final_dataset.shape)
print("Columns:", list(final_dataset.columns))

Saved final modeling dataset.
Final shape: (5252, 13)
Columns: ['Customer ID', 'total_spend', 'total_quantity', 'num_distinct_products', 'avg_price_per_item', 'first_order_date', 'country', 'is_uk', 'order_month', 'order_dayofweek', 'repeat_purchase', 'log_total_spend', 'log_total_quantity']
